# WavqWise: Weather Forecasting with Real Data
**Sense. Forecast. Alert.**

Real weather data from Open-Meteo API (free, no key). Forecast temperature, precipitation, wind for any city worldwide.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VK-Ant/wavqwise/blob/main/demos/notebooks/wavqwise_weather_forecast.ipynb)

**Data Source:** [Open-Meteo](https://api.open-meteo.com) (free, open, no API key)

**Author:** [VK-Ant](https://github.com/VK-Ant)

In [ ]:
!pip install wavqwise requests -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wavqwise import WavqPipeline, WeatherPipeline
from wavqwise.weather.loader import WeatherLoader
from wavqwise.weather.models import WeatherIndicators
from wavqwise.visualization.renderer import ResultRenderer, CLIPrinter
print('WavqWise loaded')

## 1. Load Real Weather Data
Change the city to any: Chennai, Tokyo, London, New York, Mumbai, Paris, Berlin, Dubai, Singapore...

In [ ]:
CITY = 'Chennai'  # Change to any city
DAYS = 365

weather = WeatherPipeline()
weather.load_city(CITY, days=DAYS)
print(weather.summary())
weather.data.tail()

## 2. Temperature Analysis

In [ ]:
df = weather.data
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(f'Weather Analysis: {CITY}', fontsize=16, fontweight='bold')

# Temperature
ax = axes[0,0]
ax.plot(df['date'], df['temperature_2m_mean'], color='#dc2626', linewidth=1, label='Mean')
if 'temperature_2m_max' in df.columns:
    ax.fill_between(df['date'], df['temperature_2m_min'], df['temperature_2m_max'], alpha=0.15, color='#dc2626', label='Range')
if 'heat_index' in df.columns:
    ax.plot(df['date'], df['heat_index'], color='#f59e0b', linewidth=0.8, alpha=0.6, label='Heat Index')
ax.set_title('Temperature (C)'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Precipitation
ax = axes[0,1]
if 'precipitation_sum' in df.columns:
    ax.bar(df['date'], df['precipitation_sum'], color='#2563eb', alpha=0.6, width=1)
ax.set_title('Precipitation (mm)'); ax.grid(True, alpha=0.3)

# Wind
ax = axes[1,0]
if 'windspeed_10m_max' in df.columns:
    ax.plot(df['date'], df['windspeed_10m_max'], color='#7c3aed', linewidth=0.8)
    ax.fill_between(df['date'], 0, df['windspeed_10m_max'], alpha=0.1, color='#7c3aed')
ax.set_title('Wind Speed (km/h)'); ax.grid(True, alpha=0.3)

# Humidity
ax = axes[1,1]
if 'relative_humidity_2m_mean' in df.columns:
    ax.plot(df['date'], df['relative_humidity_2m_mean'], color='#0d9488', linewidth=0.8)
ax.set_title('Humidity (%)'); ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

## 3. Temperature Forecast (30 days)

In [ ]:
# Forecast with multiple models
for model in ['moving_average', 'ema', 'naive', 'seasonal_naive']:
    fc = weather.forecast(target='temperature_2m_mean', horizon=30, model=model)
    print(f'{model}: day1={fc.forecast["temperature_2m_mean"].iloc[0]:.1f}C, day30={fc.forecast["temperature_2m_mean"].iloc[-1]:.1f}C')

## 4. Model Comparison

In [ ]:
comp = weather.compare_models(target='temperature_2m_mean',
    models=['moving_average', 'ema', 'naive', 'seasonal_naive'], horizon=14)
print(comp.to_string(index=False))
print(f'Best model: {comp.iloc[0]["model"]}')

## 5. Forecast Chart

In [ ]:
best = comp.iloc[0]['model']
forecast = weather.forecast(target='temperature_2m_mean', horizon=30, model=best)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df['date'], df['temperature_2m_mean'], color='#2563eb', linewidth=1, label='Historical')
fc = forecast.forecast
ax.plot(fc['date'], fc['temperature_2m_mean'], '--', color='#dc2626', linewidth=2, label=f'Forecast ({best})')
ax.fill_between(fc['date'], fc['temperature_2m_mean_lower'], fc['temperature_2m_mean_upper'], alpha=0.12, color='#dc2626', label='95% CI')
ax.set_title(f'{CITY} Temperature Forecast (30 days)', fontsize=14, fontweight='bold')
ax.set_ylabel('Temperature (C)'); ax.legend(); ax.grid(True, alpha=0.3)
ax.text(0.99, 0.01, 'Data: Open-Meteo (api.open-meteo.com)', transform=ax.transAxes, fontsize=8, color='#94a3b8', ha='right', va='bottom')
plt.tight_layout(); plt.show()

## 6. Multi-City Comparison

In [ ]:
cities = ['Chennai', 'Tokyo', 'London', 'New York']
fig, ax = plt.subplots(figsize=(14, 6))
for city, color in zip(cities, ['#dc2626', '#2563eb', '#059669', '#7c3aed']):
    try:
        w = WeatherPipeline()
        w.load_city(city, days=180)
        ax.plot(w.data['date'], w.data['temperature_2m_mean'], color=color, linewidth=1, label=city, alpha=0.8)
    except Exception as e:
        print(f'{city}: {e}')
ax.set_title('Temperature Comparison (180 days)', fontsize=14, fontweight='bold')
ax.set_ylabel('Temperature (C)'); ax.legend(); ax.grid(True, alpha=0.3)
ax.text(0.99, 0.01, 'Data: Open-Meteo', transform=ax.transAxes, fontsize=8, color='#94a3b8', ha='right', va='bottom')
plt.tight_layout(); plt.show()

## 7. Precipitation Forecast

In [ ]:
if 'precipitation_sum' in df.columns:
    fc_rain = weather.forecast(target='precipitation_sum', horizon=14, model='ema')
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(df['date'].tail(60), df['precipitation_sum'].tail(60), color='#2563eb', alpha=0.5, width=1, label='Historical')
    ax.bar(fc_rain.forecast['date'], fc_rain.forecast['precipitation_sum'], color='#dc2626', alpha=0.6, width=1, label='Forecast')
    ax.set_title(f'{CITY} Precipitation Forecast (14 days)', fontweight='bold')
    ax.set_ylabel('Precipitation (mm)'); ax.legend(); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

## 8. Weather Indicators

In [ ]:
if 'heat_index' in df.columns:
    print(f'Heat Index: avg={df["heat_index"].mean():.1f}C, max={df["heat_index"].max():.1f}C')
if 'dew_point' in df.columns:
    print(f'Dew Point: avg={df["dew_point"].mean():.1f}C')
if 'wind_chill' in df.columns:
    print(f'Wind Chill: min={df["wind_chill"].min():.1f}C')
if 'temp_range' in df.columns:
    print(f'Daily Temp Range: avg={df["temp_range"].mean():.1f}C')

## 9. Available Cities & Data Sources

In [ ]:
print('Available cities:', WeatherLoader.available_cities())
print('\nData source: Open-Meteo (api.open-meteo.com)')
print('Coverage: Global | History: 80+ years | Forecast: 16 days')
print('API key: Not required (free)')

---
**WavqWise** - Sense. Forecast. Alert. | [GitHub](https://github.com/VK-Ant/wavqwise) | [PyPI](https://pypi.org/project/wavqwise/)

**Data:** [Open-Meteo](https://api.open-meteo.com) (free, open, no API key)